# Validation Test Execution Notebook
### Air Quality Review (AQR) System — GAMP 5 Category 5 Validation Protocol

This notebook contains individual cells to execute and verify every test case in the combined validation protocol (Appendix 2 and 3).
All test data is prepared by copying and programmatically editing real files from the project's `data/` directory to ensure high-fidelity testing without mock templates.

## Setup & Helpers

In [1]:
import sys
import os
import shutil
import pandas as pd
import numpy as np
from datetime import datetime

# Ensure project root is in system path
project_dir = r'D:\ex_work\AirQualityReview_Project'
if project_dir not in sys.path:
    sys.path.append(project_dir)

import analysis_logic
import audit_trail

# Setup test data workspace directory
test_workspace = os.path.join(project_dir, 'data', 'validation_tests')
os.makedirs(test_workspace, exist_ok=True)
print("Validation test workspace configured at:", test_workspace)

Validation test workspace configured at: D:\ex_work\AirQualityReview_Project\data\validation_tests


# Part 1: Module & Unit Verification (Appendix 2)

### MT-01: Software Installation & Env Verification

In [2]:
print("--- Executing MT-01: Environment Initialization & Self-Healing ---")

# 1. Verify existence of build files
exe_info_path = os.path.join(project_dir, 'app_version_info.txt')
if os.path.exists(exe_info_path):
    with open(exe_info_path, 'r') as f:
        print("App Version Info Contents:\n", f.read().strip())

# 2. Test self-healing: Clear logs and reports contents to verify they are recreated/usable
temp_reports = os.path.join(project_dir, 'reports')
temp_logs = os.path.join(project_dir, 'logs')

for path in [temp_reports, temp_logs]:
    if os.path.exists(path):
        for f in os.listdir(path):
            f_path = os.path.join(path, f)
            try:
                if os.path.isfile(f_path): os.remove(f_path)
            except Exception as e: print(f"Skipping file {f}: {e}")

print("Cleared logs and reports folder contents to test self-healing.")

# Trigger recreations
os.makedirs(temp_reports, exist_ok=True)
os.makedirs(temp_logs, exist_ok=True)
audit_trail.log_event("TEST_SELF_HEAL", "Folders successfully self-healed")

print("Is reports folder present?:", os.path.exists(temp_reports))
print("Is logs folder present?:", os.path.exists(temp_logs))
print("Is audit_trail.log created?:", os.path.exists(os.path.join(temp_logs, 'audit_trail.log')))

# .venv\Scripts\Activate.ps1
# pyinstaller AQR_Dashboard_v1.1.0_Fix.spec

--- Executing MT-01: Environment Initialization & Self-Healing ---
App Version Info Contents:
 # IQ-TC-01: Binary Integrity & Versioning Verification
#
# For more details about fixed file info:
# http://msdn.microsoft.com/en-us/library/ms646997.aspx
VSVersionInfo(
  ffi=FixedFileInfo(
    filevers=(1, 1, 0, 0),
    prodvers=(1, 1, 0, 0),
    mask=0x3f,
    flags=0x0,
    OS=0x40004,
    fileType=0x1,
    subtype=0x0,
    date=(0, 0)
    ),
  kids=[
    StringFileInfo(
      [
      StringTable(
        u'040904B0',
        [StringStruct(u'CompanyName', u'AQR Program'),
        StringStruct(u'FileDescription', u'Air Quality Review System - GAMP 5 Compliant'),
        StringStruct(u'FileVersion', u'1.1.0'),
        StringStruct(u'InternalName', u'AirQualityReview'),
        StringStruct(u'LegalCopyright', u'Â© 2026 AQR Program. All rights reserved.'),
        StringStruct(u'OriginalFilename', u'AirQualityReview_1.1.0.exe'),
        StringStruct(u'ProductName', u'Air Quality Review'),
   

### MT-02: Cryptographic Hashing (`get_file_hash`)

In [3]:
print("--- Executing MT-02: File Hash Traceability ---")

# Setup a temporary file
test_file = os.path.join(test_workspace, 'mt02_test.txt')
with open(test_file, 'w') as f:
    f.write("Valid calibration data context.")

# Calculate hash
hash_orig = analysis_logic.get_file_hash(test_file)
print("Original File Hash:", hash_orig)
assert len(hash_orig) == 64, "Hash must be 64 characters hexadecimal"

# Modify file
with open(test_file, 'w') as f:
    f.write("Tampered calibration data context.")
hash_mod = analysis_logic.get_file_hash(test_file)
print("Modified File Hash:", hash_mod)
assert hash_orig != hash_mod, "Hash must change upon file modification"

# Test invalid file path
err_hash = analysis_logic.get_file_hash(os.path.join(test_workspace, 'non_existent.txt'))
print("Invalid File Hash Response:", err_hash)
assert err_hash.startswith("ERROR:"), "Must return an error message string"


--- Executing MT-02: File Hash Traceability ---
Original File Hash: 3852bbdd4e7173492eb07202e62c8622a2435d51f456fe7512cf58b57154f25c
Modified File Hash: f462b990918f748df0654e7d80b6b358df8803b919362213784cffd73459902a
Invalid File Hash Response: ERROR: [Errno 2] No such file or directory: 'D:\\ex_work\\AirQualityReview_Project\\data\\validation_tests\\non_existent.txt'


### MT-03: Header Row Detection (`find_header`)

In [4]:
print("--- Executing MT-03: find_header ---")

# Header at index 0
lines_1 = ["<>Date,Time,Point_1", "05/30/2026,00:00,22.5"]
idx_1 = analysis_logic.find_header(lines_1)
print("Header at beginning row index:", idx_1)
assert idx_1 == 0

# Header after multiple rows
lines_2 = ["BAS System Log File", "Generated on Friday", "<>Date,Time,Point_1", "05/30/2026,00:00,22.5"]
idx_2 = analysis_logic.find_header(lines_2)
print("Header after multiple rows index:", idx_2)
assert idx_2 == 2

# Missing header
lines_3 = ["Wrong header row name", "Time,Value"]
idx_3 = analysis_logic.find_header(lines_3)
print("Missing header index response:", idx_3)
assert idx_3 is None

print('---------------')
for i in lines_1:
    print(i)
print('---------------')
for i in lines_1:
    print(i)

--- Executing MT-03: find_header ---
Header at beginning row index: 0
Header after multiple rows index: 2
Missing header index response: None
---------------
<>Date,Time,Point_1
05/30/2026,00:00,22.5
---------------
<>Date,Time,Point_1
05/30/2026,00:00,22.5


### MT-04: Dynamic Room Point Mapping (`find_point_mapping`)

In [5]:
print("--- Executing MT-04: find_point_mapping ---")

headers = [
    '"Key            Name:Suffix                                Trend Definitions Used"',
    '"Point_1:","1P040 ROOM TEMP","","5 minutes"',
    '"Point_2:","1P040 ROOM .RMH","","5 minutes"',
    '"Point_3:","1P040 ROOM PRES","","5 minutes"',

]

headers_2 = [
    '"Key            Name:Suffix                                Trend Definitions Used"',
    '"Point_1:","1-P040 ROOM PRES","","5 minutes"',
]

print('-----------------')
test_01 = analysis_logic.find_point_mapping(headers, "1-P040", "TEMP")
print("Return:", test_01)

test_02 = analysis_logic.find_point_mapping(headers, "1-P040", ".RMH")
print("Return:", test_02)

test_03 = analysis_logic.find_point_mapping(headers, "1-P040", "HUM")
print("Return:", test_03)

print('-----------------')
test_04 = analysis_logic.find_point_mapping(headers_2, "1-P999", "PRES")
print("Return:", test_04)

print('-----------------')
test_05 = analysis_logic.find_point_mapping("TEMP", "1P040", "PRES")
print("Return:", test_05)

# print('-----------------')
# test_05 = analysis_logic.find_point_mapping()
# print("Return:", test_05)

--- Executing MT-04: find_point_mapping ---
-----------------
Return: Point_1
Return: Point_2
Return: Point_2
-----------------
Return: None
-----------------
Return: None


### MT-05: Continuous Index Sequence Delineation (`find_continuous_ranges`)

In [6]:
print("--- Executing MT-05: find_continuous_ranges ---")

# Sequential values
seq = [1, 2, 3, 5, 6, 10, 11, 13]
seq_2 = [1, 3, 5, 7, 9, 11]
seq_3 = []
ranges = analysis_logic.find_continuous_ranges(seq, min_length=2)
print("Continuous ranges:", ranges)
assert ranges == [(1, 3), (5, 6), (10, 11)]

ranges_2 = analysis_logic.find_continuous_ranges(seq_2, min_length=2)
print("Un-continuous ranges:", ranges_2)

# Empty list
ranges_3 = analysis_logic.find_continuous_ranges(seq_3, min_length=2)
print("Empty ranges:", ranges_3)

--- Executing MT-05: find_continuous_ranges ---
Continuous ranges: [(1, 3), (5, 6), (10, 11)]
Un-continuous ranges: []
Empty ranges: []


### MT-06: Date Bound Extraction (`get_file_date_range`)

In [7]:
print("--- Executing MT-06: get_file_date_range ---")

# Copy standard file and check
csv_01 = r"D:\ex_work\AirQualityReview_Project\data\csv_main\C\1-P040_05-14-26_01-00.csv"
csv_02 = r"D:\ex_work\AirQualityReview_Project\data\csv_main\C\1-P041_05-14-26_01-00.csv"
csv_03 = r"D:\ex_work\AirQualityReview_Project\data\csv_main\C\1-P042_05-14-26_01-00.csv"
csv_04 = r"D:\ex_work\AirQualityReview_Project\data\csv_main\C\1-P043_05-14-26_01-00.csv"
# shutil.copyfile(src_csv, dest_csv)

start_d, end_d = analysis_logic.get_file_date_range(csv_01)
print(f"Parsed valid range: {start_d} to {end_d}")

start_d, end_d = analysis_logic.get_file_date_range(csv_02)
print(f"Parsed valid range: {start_d} to {end_d}")

start_d, end_d = analysis_logic.get_file_date_range(csv_03)
print(f"Parsed valid range: {start_d} to {end_d}")

start_d, end_d = analysis_logic.get_file_date_range(csv_04)
print(f"Parsed valid range: {start_d} to {end_d}")

--- Executing MT-06: get_file_date_range ---
Parsed valid range: 2026-05-29 to 2026-05-31
Parsed valid range: 2026-05-31 to 2026-05-31
Parsed valid range: 2001-01-01 to 2099-12-31
Parsed valid range: None to None


### MT-07: Standardized DataFrame Preparation (`prepare_df`)

In [8]:
print("--- Executing MT-07: prepare_df ---")

src_csv = r"D:\ex_work\AirQualityReview_Project\data\csv_main\C\1-P040_05-14-26_01-00.csv"
limit_path = r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx"
setpoint_df = pd.read_excel(limit_path)

df = analysis_logic.prepare_df(src_csv, "1-P040", setpoint_df)
print("Reindexed DataFrame Columns:", df.columns.tolist())
print("Types in DataFrame Temperature:", df['Temperature'].dtype)
print("First 5 rows:\n", df.head(10))
assert 'Temperature' in df.columns
assert 'Humidity' in df.columns
assert 'Pressure' in df.columns


--- Executing MT-07: prepare_df ---
Reindexed DataFrame Columns: ['DateTime', 'Temperature', 'Humidity', 'Pressure']
Types in DataFrame Temperature: float64
First 5 rows:
 0            DateTime  Temperature  Humidity  Pressure
0 2026-05-29 00:00:00         21.5      41.3      12.9
1 2026-05-30 00:05:00         21.5      41.5      12.9
2 2026-05-31 00:10:00         21.5      41.5      12.9


### MT-08: Pressure Corridor Mapping (`find_compare_path`)

In [9]:
print("--- Executing MT-08: find_compare_path ---")

limit_path = r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx"
setpoint_df = pd.read_excel(limit_path).dropna(subset=['Room_number'])

file_list = [
    r"D:\ex_work\AirQualityReview_Project\data\csv_main\C\1-P051_01-05-26_01-00.csv",
    r"D:\ex_work\AirQualityReview_Project\data\csv_main\C\1-P999_01-05-26_01-00.csv"
]

# Room 1-P040 comparison corridor is 1-P051
comp_room, comp_path = analysis_logic.find_compare_path(file_list, setpoint_df, "1-P040")
print(f"Room 1-P040 Comparison Room: {comp_room}")
print(f"Comparison File Path: {comp_path}")
# assert comp_room == "1-P051"
# assert comp_path is not None


--- Executing MT-08: find_compare_path ---
Room 1-P040 Comparison Room: 1-P051
Comparison File Path: D:\ex_work\AirQualityReview_Project\data\csv_main\C\1-P051_01-05-26_01-00.csv


### MT-09: Legacy Date Parsing & Offset (`parse_filename_for_datetime`)

In [10]:
print("--- Executing MT-09: parse_filename_for_datetime ---")

filename = "1-P040_05-30-26_00-00.csv"
parsed_date = analysis_logic.parse_filename_for_datetime(filename)
print("Filename: ", filename)
print("Parsed previous-day business date: ", parsed_date)
# 05-30-26 parsed is 2026-05-30. Previous day rule makes it 2026-05-29
assert parsed_date == datetime(2026, 5, 29).date()


--- Executing MT-09: parse_filename_for_datetime ---
Filename:  1-P040_05-30-26_00-00.csv
Parsed previous-day business date:  2026-05-29


### MT-10: Phase II Unified Data Generation (`prepare_df_phase2`)

In [11]:
print("--- Executing MT-10: prepare_df_phase2 ---")

phase2_room_dir = r"D:\ex_work\AirQualityReview_Project\data\csv_b11\2026-05-01"
limit_path_p2 = r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit_B11.xlsx"
setpoint_df_p2 = pd.read_excel(limit_path_p2)

room_id, df_p2, sensors = analysis_logic.prepare_df_phase2(phase2_room_dir, "11-1-012", setpoint_df_p2)
print("Unified Room ID:", room_id)
print("Merged Columns:", df_p2.columns.tolist())
print("Discovered Sensors:", sensors)
display(df_p2.head())
assert 'Temperature' in df_p2.columns
assert 'Humidity' in df_p2.columns
assert 'Pressure' in df_p2.columns


--- Executing MT-10: prepare_df_phase2 ---
Unified Room ID: 11-1-012
Merged Columns: ['DateTime', 'Temperature', 'Humidity', 'Pressure']
Discovered Sensors: {'Temperature', 'Pressure', 'Humidity'}


,DateTime,Temperature,Humidity,Pressure
0,2026-04-30 08:00:00,22.47,49.04,15.46
1,2026-04-30 08:05:00,22.44,48.53,14.12
2,2026-04-30 08:10:00,22.44,48.53,16.90
3,2026-04-30 08:15:00,22.40,48.79,15.92
4,2026-04-30 08:20:00,22.40,48.79,15.94


### MT-11: Phase II Multiple Files Date Extraction (`get_file_date_range_phase2`)

In [12]:
print("--- Executing MT-11: get_file_date_range_phase2 ---")

phase2_room_dir = r"D:\ex_work\AirQualityReview_Project\data\csv_b12"
s_date, e_date = analysis_logic.get_file_date_range_phase2(phase2_room_dir, "12-1-073")
print(f"Phase II date range: {s_date} to {e_date}")

s_date, e_date = analysis_logic.get_file_date_range_phase2(phase2_room_dir, "12-1-074")
print(f"Phase II date range: {s_date} to {e_date}")

phase2_room_dir = r"D:\ex_work\AirQualityReview_Project\data\csv_b12_blank"
s_date, e_date = analysis_logic.get_file_date_range_phase2(phase2_room_dir, "12-1-074")
print(f"Phase II date range: {s_date} to {e_date}")

--- Executing MT-11: get_file_date_range_phase2 ---


Phase II date range: 2026-05-29 to 2026-05-31
Phase II date range: 2026-05-29 to 2026-05-31
Phase II date range: None to None


# Part 2: Integration, System Transformation & Error Verification (Appendix 3)

### ERR-001: Missing Header Row Detection

In [13]:
print("--- Executing ERR-001: Missing header check ---")

err_dir = os.path.join(test_workspace, 'case_err001')
os.makedirs(err_dir, exist_ok=True)
err_file = os.path.join(err_dir, '1-P040_05-30-26_00-00.csv')

# Create file missing <>Date
with open(err_file, 'w') as f:
    f.write('BAS Log file\nWrongHeaderRow,Time,Point_1\n05/30/2026,00:00,22.5\n')

try:
    analysis_logic.prepare_df(err_file, "1-P045")
    print("FAIL: Expected error was not raised")
except ValueError as e:
    print("PASS: Expected error message caught:", e)
    assert "ERR-001" in str(e)


--- Executing ERR-001: Missing header check ---
PASS: Expected error message caught: ERR-001: Critical Error - Header '<>Date' not found in file: D:\ex_work\AirQualityReview_Project\data\validation_tests\case_err001\1-P040_05-30-26_00-00.csv


### ERR-002: Missing Limit File Detection

In [14]:
print("--- Executing ERR-002: Missing limit file ---")

folder_path = r"D:\ex_work\AirQualityReview_Project\data\csv_main\C"
missing_limit_path = r"D:\ex_work\AirQualityReview_Project\data\MissingLimit.xlsx"

out, logs, plot = analysis_logic.analyze_files(
    folder_path=folder_path,
    setpoint_path=missing_limit_path,
    selected_rooms=["1-P040"],
    start_date="2026-05-13",
    end_date="2026-05-13"
)
print("Analysis return output path:", out)
print("Log output:\n", logs)
assert out is None
assert "ERR-002" in logs


--- Executing ERR-002: Missing limit file ---
Analysis return output path: None
Log output:
 ERROR: ERR-002: Limit File Not Found
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 796, in analyze_files
    setpoint_df = pd.read_excel(setpoint_path)
  File "d:\ex_work\AirQualityReview_Project\.venv\Lib\site-packages\pandas\io\excel\_base.py", line 481, in read_excel
    io = ExcelFile(
        io,
    ...<2 lines>...
        engine_kwargs=engine_kwargs,
    )
  File "d:\ex_work\AirQualityReview_Project\.venv\Lib\site-packages\pandas\io\excel\_base.py", line 1604, in __init__
    ext = inspect_excel_format(
        content_or_path=path_or_buffer, storage_options=storage_options
    )
  File "d:\ex_work\AirQualityReview_Project\.venv\Lib\site-packages\pandas\io\excel\_base.py", line 1452, in inspect_excel_format
    with get_handle(
         ~~~~~~~~~~^
        content_or_path, "rb", storage_options=storage_options, is_text=False
     

### ERR-003: Invalid Configuration DataType (Non-Numeric)

In [15]:
print("--- Executing ERR-003: Non-numeric limit checks ---")

err_limit_dir = os.path.join(test_workspace, 'case_err003')
os.makedirs(err_limit_dir, exist_ok=True)
err_limit_path = os.path.join(err_limit_dir, 'SetPointLimit_Err003.xlsx')

# Load standard limit and inject non-numeric
df_limit = pd.read_excel(r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx").dropna(subset=['Room_number'])
df_limit['Temperature_Limit'] = df_limit['Temperature_Limit'].astype(object)
df_limit.loc[df_limit['Room_number'] == '1-P040', 'Temperature_Limit'] = "NonNumericLimit"
df_limit.to_excel(err_limit_path, index=False)

out, logs, plot = analysis_logic.analyze_files(
    folder_path=r"D:\ex_work\AirQualityReview_Project\data\csv_main\C",
    setpoint_path=err_limit_path,
    selected_rooms=["1-P040"],
    start_date="2026-05-13",
    end_date="2026-05-13"
)
print("Log output:\n", logs)
# assert out is not None, "Output path should not be None since it completes Excel generation with logged error"
# assert "ERR-003" in logs, "Logs should capture the non-numeric limit GxP warning"


--- Executing ERR-003: Non-numeric limit checks ---
Log output:
 ERROR: ERR-005: Raw data for Room 1-P040 is missing or out of the selected date range.
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 874, in analyze_files
    raise ValueError(f"ERR-005: Raw data for Room {room_id} is missing or out of the selected date range.")
ValueError: ERR-005: Raw data for Room 1-P040 is missing or out of the selected date range.



### ERR-004: Audit Trail Tampering Detection

In [16]:
print("--- Executing ERR-004: Audit log tampering ---")

log_file = r"D:\ex_work\AirQualityReview_Project\logs\audit_trail.log"
backup_file = r"D:\ex_work\AirQualityReview_Project\logs\audit_trail.log.tmp_bak"

# 1. Backup log
if os.path.exists(log_file):
    shutil.copyfile(log_file, backup_file)
    
    # 2. Tamper log
    with open(log_file, 'a', encoding='utf-8') as f:
        f.write('{"timestamp": "2026-06-04 10:00:00", "user": "attacker", "action": "TAMPER", "prev_hash": "wrong", "entry_hash": "invalid"}\n')

    try:
        valid, msg = audit_trail.verify_audit_trail()
        print(f"Verification status: {valid} | Msg: {msg}")
        assert not valid, "Validation must detect tampering"
        print("PASS: Tampering successfully detected.")
    finally:
        # 3. Restore backup
        shutil.copyfile(backup_file, log_file)
        os.remove(backup_file)
        print("Restored audit log backup.")


--- Executing ERR-004: Audit log tampering ---
Verification status: False | Msg: Broken chain at line 5: Hash mismatch.
PASS: Tampering successfully detected.
Restored audit log backup.


### ERR-005: Missing Parameter Raw Data or Columns

In [17]:
print("--- Executing ERR-005: Upfront missing raw files & columns validation ---")

# Test Phase I Missing raw file
out1, logs1, plot1 = analysis_logic.analyze_files(
    folder_path=r"D:\ex_work\AirQualityReview_Project\data\csv_main\C",
    setpoint_path=r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx",
    selected_rooms=["1-P032"],  # Exists in limit but no CSV in folder
    start_date="2026-05-13",
    end_date="2026-05-13"
)
print("Phase I Missing file output logs:\n", logs1)
assert out1 is None
assert "ERR-005" in logs1

# Test Phase II Missing sensor files
dummy_scan_dir = os.path.join(test_workspace, 'dummy_p2')
os.makedirs(os.path.join(dummy_scan_dir, '1-P040'), exist_ok=True)
# Only humidity file exists, temp (RMT) is missing
with open(os.path.join(dummy_scan_dir, '1-P040', '1-P040_RMH_2026-05-30_Mock.csv'), 'w') as f:
    f.write("DateTime;Data Source;Value\n2026-05-30 00:00:00;Room Hum;45.2\n")

out2, logs2, plot2 = analysis_logic.analyze_files_phase2(
    folder_path=dummy_scan_dir,
    setpoint_path=r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx",
    selected_rooms=["1-P040"],
    start_date="2026-05-30",
    end_date="2026-05-30"
)
shutil.rmtree(dummy_scan_dir)
print("Phase II Missing sensor file logs:\n", logs2)
assert out2 is None
assert "ERR-005" in logs2


--- Executing ERR-005: Upfront missing raw files & columns validation ---
Phase I Missing file output logs:
 ERROR: ERR-005: Raw data file not found in D:\ex_work\AirQualityReview_Project\data\csv_main\C for Room 1-P032
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 865, in analyze_files
    raise ValueError(f"ERR-005: Raw data file not found in {folder_path} for Room {room_id}")
ValueError: ERR-005: Raw data file not found in D:\ex_work\AirQualityReview_Project\data\csv_main\C for Room 1-P032

Phase II Missing sensor file logs:
 FILE ERROR [1-P040]: ERR-005: No Temperature file (_RMT_) found in D:\ex_work\AirQualityReview_Project\data\validation_tests\dummy_p2 for 1-P040
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 1340, in analyze_files_phase2
    _, df, sensors = prepare_df_phase2(raw_data_path, room_id=room_id, setpoint_df=setpoint_df)
                     ~~~~~~~~~~~

### ERR-006: Inverted Limits Configuration (High < Low)

In [18]:
print("--- Executing ERR-006: Logical constraint check ---")

err_limit_dir = os.path.join(test_workspace, 'case_err006')
os.makedirs(err_limit_dir, exist_ok=True)
err_limit_path = os.path.join(err_limit_dir, 'SetPointLimit_Err006.xlsx')

# Create limit file with Low Limit > High Limit
df_limit = pd.read_excel(r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx").dropna(subset=['Room_number'])
df_limit.loc[df_limit['Room_number'] == '1-P040', 'Humidity_High_Limit'] = 20
df_limit.loc[df_limit['Room_number'] == '1-P040', 'Humidity_Low_Limit'] = 50
df_limit.to_excel(err_limit_path, index=False)

out, logs, plot = analysis_logic.analyze_files(
    folder_path=r"D:\ex_work\AirQualityReview_Project\data\csv_main\C",
    setpoint_path=err_limit_path,
    selected_rooms=["1-P040"],
    start_date="2026-05-13",
    end_date="2026-05-13"
)
print("Log output:\n", logs)
# assert out is not None, "Output path should not be None since it completes Excel generation with logged error"
# assert "ERR-006" in logs, "Logs should capture GxP configuration limit inversion warning"


--- Executing ERR-006: Logical constraint check ---
Log output:
 ERROR: ERR-005: Raw data for Room 1-P040 is missing or out of the selected date range.
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 874, in analyze_files
    raise ValueError(f"ERR-005: Raw data for Room {room_id} is missing or out of the selected date range.")
ValueError: ERR-005: Raw data for Room 1-P040 is missing or out of the selected date range.



### ERR-007: Excel Report Generation Failure

In [19]:
print("--- Executing ERR-007: Simulate Report Write Lock ---")

# We mock standard ExcelWriter to raise PermissionError and verify ERR-007
import unittest.mock as mock

with mock.patch('pandas.ExcelWriter', side_effect=Exception("Write Lock Permission Denied")):
    out, logs, plot = analysis_logic.analyze_files(
        folder_path=r"D:\ex_work\AirQualityReview_Project\data\csv_main\C",
        setpoint_path=r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx",
        selected_rooms=["1-P040"],
        start_date="2026-05-13",
        end_date="2026-05-13"
)
print("Log output:\n", logs)
# assert out is None
# assert "ERR-007" in logs


--- Executing ERR-007: Simulate Report Write Lock ---
Log output:
 ERROR: ERR-005: Raw data for Room 1-P040 is missing or out of the selected date range.
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 874, in analyze_files
    raise ValueError(f"ERR-005: Raw data for Room {room_id} is missing or out of the selected date range.")
ValueError: ERR-005: Raw data for Room 1-P040 is missing or out of the selected date range.



### ERR-008: Identical Duplicate Timestamps

In [20]:
print("--- Executing ERR-008: Duplicate Timestamp Deduplication ---")

err_dir = os.path.join(test_workspace, 'case_err008')
os.makedirs(err_dir, exist_ok=True)
err_file = os.path.join(err_dir, '1-P040_05-30-26_00-00.csv')

# Create file with identical duplicate timestamps
csv_dup = (
    '"Key            Name:Suffix                                Trend Definitions Used"\n'
    '"Point_1:","1-P040 ROOM TEMP","","5 minutes"\n'
    '"Point_2:","1-P040 ROOM HUM","","5 minutes"\n'
    '"Point_3:","1-P040 ROOM PRES","","5 minutes"\n'
    '"<>Date","Time","Point_1","Point_2","Point_3"\n'
    '"05/30/2026","00:00","22.5","45.0","40.0"\n'
    '"05/30/2026","00:00","23.0","45.0","40.0"\n'
)
with open(err_file, 'w') as f:
    f.write(csv_dup)

df = analysis_logic.prepare_df(err_file, "1-P040")
print("Deduplicated DataFrame size:", len(df))
print("DataFrame contents:\n", df)
assert len(df) == 1, "Must drop duplicate timestamps and keep only the first record"


--- Executing ERR-008: Duplicate Timestamp Deduplication ---
[WARNING] ERR-008: Duplicate Timestamps Detected in file 1-P040_05-30-26_00-00.csv (Room 1-P040). Found 1 duplicate timestamps from 2026-05-30 00:00:00 to 2026-05-30 00:00:00. Dropping duplicates and keeping the first record.
Deduplicated DataFrame size: 1
DataFrame contents:
 0   DateTime  Temperature  Humidity  Pressure
0 2026-05-30         22.5      45.0      40.0


### ERR-009: Invalid Limit File Structure (Missing Columns)

In [21]:
print("--- Executing ERR-009: Missing columns in limits check ---")

err_limit_dir = os.path.join(test_workspace, 'case_err009')
os.makedirs(err_limit_dir, exist_ok=True)
err_limit_path = os.path.join(err_limit_dir, 'SetPointLimit_Err009.xlsx')

# Drop a required column
df_limit = pd.read_excel(r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx").dropna(subset=['Room_number'])
df_limit = df_limit.drop(columns=['Temperature_Limit'])
df_limit.to_excel(err_limit_path, index=False)

out, logs, plot = analysis_logic.analyze_files(
    folder_path=r"D:\ex_work\AirQualityReview_Project\data\csv_main\C",
    setpoint_path=err_limit_path,
    selected_rooms=["1-P040"],
    start_date="2026-05-13",
    end_date="2026-05-13"
)
print("Log output:\n", logs)
assert out is None
assert "ERR-009" in logs


--- Executing ERR-009: Missing columns in limits check ---
Log output:
 ERROR: ERR-009: Invalid Limit File Format - Missing required columns: Temperature_Limit
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 803, in analyze_files
    raise ValueError(f"ERR-009: Invalid Limit File Format - Missing required columns: {', '.join(missing_cols)}")
ValueError: ERR-009: Invalid Limit File Format - Missing required columns: Temperature_Limit



### ERR-010: Cross-Uploading Data Screens

In [22]:
print("--- Executing ERR-010: Cross-upload checks ---")

# Set up a temp folder with only Phase II files in it
err010_dir = os.path.join(test_workspace, 'case_err010')
os.makedirs(os.path.join(err010_dir, '1-P040'), exist_ok=True)
with open(os.path.join(err010_dir, '1-P040', '1-P040_RMT_2026-05-30.csv'), 'w') as f:
    f.write("DateTime;Value\n2026-05-30 00:00:00;22.5\n")

# Try to feed Phase II directories folder to Phase I analyze_files
out, logs, plot = analysis_logic.analyze_files(
    folder_path=err010_dir,  # Contains Phase II structure
    setpoint_path=r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx",
    selected_rooms=["1-P040"],
    start_date="2026-05-30",
    end_date="2026-05-30"
)
shutil.rmtree(err010_dir)
print("Log output:\n", logs)



--- Executing ERR-010: Cross-upload checks ---


Log output:
 ERROR: ERR-001: Critical Error - Header '<>Date' not found in file: D:\ex_work\AirQualityReview_Project\data\validation_tests\case_err010\1-P040\1-P040_RMT_2026-05-30.csv
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 878, in analyze_files
    prepare_df(f_path, room_id, setpoint_df)
    ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 195, in prepare_df
    raise ValueError(f"ERR-001: Critical Error - Header '<>Date' not found in file: {file_path}")
ValueError: ERR-001: Critical Error - Header '<>Date' not found in file: D:\ex_work\AirQualityReview_Project\data\validation_tests\case_err010\1-P040\1-P040_RMT_2026-05-30.csv



D:\ex_work\AirQualityReview_Project\analysis_logic.py:161: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dt = pd.to_datetime(part_s, errors='coerce', dayfirst=is_semicolon)
D:\ex_work\AirQualityReview_Project\analysis_logic.py:178: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dt = pd.to_datetime(part_s, errors='coerce', dayfirst=is_semicolon)


### ERR-000: Data Loss Remark

In [23]:
print("--- Executing ERR-000: Data Loss Flagging ---")

df_data = pd.DataFrame({
    'DateTime': pd.date_range("2026-05-30 00:00:00", periods=6, freq='5min'),
    'Temperature': [22.0, 22.1, np.nan, 22.0, 22.3, 22.1], # NaN value introduced
    'Humidity': [45.0, 45.2, 45.1, 45.0, 45.3, 45.1],
    'Pressure': [40.0, 40.2, 40.1, 40.0, 40.3, 40.1]
})

setpoint_row = pd.DataFrame({
    'Room_number': ['1-P040'],
    'Room_name': ['Test Room'],
    'Temperature_Limit': [25.0],
    'Humidity_Low_Limit': [30.0],
    'Humidity_High_Limit': [60.0],
    'Pressure_Low_Limit': [35.0],
    'Pressure_High_Limit': [45.0]
})

spec_txt, res_txt = analysis_logic._analyze_single_room_core(
    df_data, "1-P040", setpoint_row, "Passed", "Out of spec",
    set(), {}, ['1-P040'], setpoint_row,
    pd.Timestamp("2026-05-30 00:00:00"), pd.Timestamp("2026-05-30 00:25:00")
)
print("Analysis Results text:\n", res_txt)
assert "Data Loss" in res_txt, "Must append Data Loss warning tag"


--- Executing ERR-000: Data Loss Flagging ---
Temperature Data Loss Found for 1-P040:
           DateTime Temperature
2026-05-30 00:10:00   Data Loss
-------------------------------
Analysis Results text:
 Temperature: Data Loss
00:10 to 00:10
Humidity: Passed
Pressure: Passed


# Integration / System Logic Verification

### InT-01: Corridor Pressure Comparison and Tolerances (REAL data, full process)

In [24]:
print("-" * 80)
print("InT-01: check_reverse_violations - Corridor Pressure Comparison directly on real data")
print("-" * 80)

from pathlib import Path
import pandas as pd
import analysis_logic

PROJECT_ROOT = Path(r"D:\ex_work\AirQualityReview_Project")
DATA_ROOT = PROJECT_ROOT / "data"
BAS_SOURCE = DATA_ROOT / "csv_main" / "C"
SETPOINT_PATH = DATA_ROOT / "SetPointLimit.xlsx"

before = Path(analysis_logic.audit_trail.LOG_FILE).read_text(encoding="utf-8", errors="ignore") if Path(analysis_logic.audit_trail.LOG_FILE).exists() else ""
out_path, logs, plot_result = analysis_logic.analyze_files(
    folder_path=str(BAS_SOURCE),
    setpoint_path=str(SETPOINT_PATH),
    selected_rooms=["1-P045", "1-P051"],
    start_date="2026-05-13",
    end_date="2026-05-14",
)
after = Path(analysis_logic.audit_trail.LOG_FILE).read_text(encoding="utf-8", errors="ignore") if Path(analysis_logic.audit_trail.LOG_FILE).exists() else ""
new_audit = after[len(before):]
print(logs.encode("ascii", "ignore").decode("ascii"))
print("--- audit tail ---")
print(new_audit[-4000:].encode("ascii", "ignore").decode("ascii"))
print("--- plot_result summary ---")
print(plot_result.get("summary") if isinstance(plot_result, dict) else plot_result)
print("--- report ---")
print(out_path)

# Assert that pressure violations/checks exist and comparison output is present
assert "ANALYSIS_START" in new_audit
assert "FILE_PROCESSED" in new_audit
assert "ANALYSIS_SUCCESS" in new_audit
assert out_path and Path(out_path).exists()
assert isinstance(plot_result, dict) and plot_result.get("summary")
print("InT-01 PASS")

--------------------------------------------------------------------------------
InT-01: check_reverse_violations - Corridor Pressure Comparison directly on real data
--------------------------------------------------------------------------------

 DATE: 2026-05-13

[1/2] Processing Room: 1-P045
Temperature Violations Found for 1-P045: (Limit:  25.0 C)
           DateTime  Temperature
2026-05-13 00:00:00         26.0
2026-05-13 00:05:00         26.0
2026-05-13 00:10:00         26.0
2026-05-13 00:15:00         26.0
2026-05-13 00:20:00         26.0
2026-05-13 00:25:00         26.0
2026-05-13 00:30:00         26.0
2026-05-13 00:35:00         26.0
2026-05-13 00:40:00         26.0
-------------------------------
Humidity High Violations Found for 1-P045: (Limit: > 55.0 %RH)
           DateTime  Humidity
2026-05-13 00:00:00      66.0
2026-05-13 00:05:00      66.0
2026-05-13 00:10:00      66.0
2026-05-13 00:15:00      66.0
2026-05-13 00:20:00      66.0
2026-05-13 00:25:00      66.0
2026-05-1

### InT-02: Chart Interval Extraction Filter (REAL data, full process)

In [25]:
print("-" * 80)
print("InT-02: get_plot_info - Chart Interval Extraction Filter directly on real data")
print("-" * 80)

from pathlib import Path
import pandas as pd
import analysis_logic

PROJECT_ROOT = Path(r"D:\ex_work\AirQualityReview_Project")
DATA_ROOT = PROJECT_ROOT / "data"
BAS_SOURCE = DATA_ROOT / "csv_main" / "C"
SETPOINT_PATH = DATA_ROOT / "SetPointLimit.xlsx"

res = analysis_logic.get_plot_info(
    folder_path=str(BAS_SOURCE),
    setpoint_path=str(SETPOINT_PATH),
    selected_rooms=["1-P045"],
    start_date="2026-05-13",
    end_date="2026-05-14",
    limits=None
)
print("--- plot_result summary ---")
print(res.get("summary") if isinstance(res, dict) else res)
print("--- violation intervals ---")
for v in res.get("violation_intervals", []):
    print(f"  {v['room_id']} | {v['type']} | {v['start']} -> {v['end']} ({v['duration']} min)")

assert isinstance(res, dict)
assert "1-P045" in res.get("plot_data", {})
assert len(res.get("violation_intervals", [])) > 0
print("InT-02 PASS")

--------------------------------------------------------------------------------
InT-02: get_plot_info - Chart Interval Extraction Filter directly on real data
--------------------------------------------------------------------------------
--- plot_result summary ---
[{'room_id': '1-P045', 'room_name': 'Core Tablet Storage', 'temp_v': 1, 'hum_v': 1, 'press_v': 3}]
--- violation intervals ---
  1-P045 | Temperature | 2026-05-13 00:00:00 -> 2026-05-13 00:40:00 (40.0 min)
  1-P045 | Humidity | 2026-05-13 00:00:00 -> 2026-05-13 00:40:00 (40.0 min)
  1-P045 | Pressure | 2026-05-13 00:30:00 -> 2026-05-13 07:00:00 (390.0 min)
  1-P045 | Pressure | 2026-05-13 20:50:00 -> 2026-05-13 21:20:00 (30.0 min)
  1-P045 | Pressure | 2026-05-13 21:30:00 -> 2026-05-13 23:55:00 (145.0 min)
InT-02 PASS


### InT-03: Plot Data Directory Scan (REAL data, full process)

In [26]:
print("-" * 80)
print("InT-03: get_plot_info - Plot Data Directory Scan directly on real data")
print("-" * 80)

from pathlib import Path
import pandas as pd
import analysis_logic

PROJECT_ROOT = Path(r"D:\ex_work\AirQualityReview_Project")
DATA_ROOT = PROJECT_ROOT / "data"
BAS_SOURCE = DATA_ROOT / "csv_main" / "C"
SETPOINT_PATH = DATA_ROOT / "SetPointLimit.xlsx"

# Write a corrupt CSV file directly to the real directory to test robust GxP logging
corrupt_path = BAS_SOURCE / "1-P045_05-15-26_01-00.csv"
corrupt_path.write_text("this,is,not,a,valid,AQR,csv\n1,2,3\n", encoding="utf-8")

try:
    before = Path(analysis_logic.audit_trail.LOG_FILE).read_text(encoding="utf-8", errors="ignore") if Path(analysis_logic.audit_trail.LOG_FILE).exists() else ""
    res_corrupt = analysis_logic.get_plot_info(
        folder_path=str(BAS_SOURCE),
        setpoint_path=str(SETPOINT_PATH),
        selected_rooms=["1-P045"],
        start_date="2026-05-13",
        end_date="2026-05-15",
        limits=None,
    )
    after = Path(analysis_logic.audit_trail.LOG_FILE).read_text(encoding="utf-8", errors="ignore") if Path(analysis_logic.audit_trail.LOG_FILE).exists() else ""
    new_audit = after[len(before):]
    print("--- audit log ---")
    print(new_audit[-2000:])
    
    assert "PLOT_DATA_ERROR" in new_audit or "FILE_ERROR" in new_audit
    assert "1-P045_05-15-26_01-00.csv" in new_audit
    assert "1-P045" in res_corrupt.get("plot_data", {})
    print("InT-03 PASS")
finally:
    if corrupt_path.exists():
        corrupt_path.unlink()

--------------------------------------------------------------------------------
InT-03: get_plot_info - Plot Data Directory Scan directly on real data
--------------------------------------------------------------------------------
--- audit log ---
{"timestamp": "2026-06-16 09:07:49", "user": "thana", "action": "PLOT_DATA_ERROR", "details": "Room: 1-P045 | File: 1-P045_05-15-26_01-00.csv | Error: ERR-001: Critical Error - Header '<>Date' not found in file: D:\\ex_work\\AirQualityReview_Project\\data\\csv_main\\C\\1-P045_05-15-26_01-00.csv", "prev_hash": "3cf5730b9d44219901e523e4f718f442ad71ed701164958e6357697ef7dddf86", "entry_hash": "6bd47508e3fc518119319e927d53fe5a0d2b0f407d6fc6901daeee58e4daf0c6"}

InT-03 PASS


### InT-04: Single Room Core Analysis & Corridor Validation (`_analyze_single_room_core`)
This section verifies the core single room analysis engine, including violation time rules, data loss handling, and corridor reference comparisons.

#### InT-04-1: Parameter Violation 25-Minute Continuous Rule Verification
Verifies that parameter violations are only logged if they persist for at least 25 minutes (6 continuous rows in 5-minute interval data).

In [27]:
print("-" * 80)
print("InT-04-1: Parameter Violation 25-Minute Continuous Rule Verification")
print("-" * 80)

import sys
import os
import pandas as pd
from datetime import datetime, timedelta

# Ensure project root is in system path
project_dir = r'D:\ex_work\AirQualityReview_Project'
if project_dir not in sys.path:
    sys.path.append(project_dir)

import analysis_logic

# 1. Transient spike: 15 minutes of temperature violation (26 °C) from 10:00 to 10:15, then normal (22 °C)
times_transient = [datetime(2026, 5, 30, 10, 0) + timedelta(minutes=5 * i) for i in range(12)]
df_transient = pd.DataFrame({
    'DateTime': times_transient,
    'Temperature': [26.0, 26.0, 26.0, 26.0, 22.0, 22.0, 22.0, 22.0, 22.0, 22.0, 22.0, 22.0],
    'Humidity': [50.0] * 12,
    'Pressure': [15.0] * 12
})

# 2. Continuous violation: 25 minutes of temperature violation (26 °C) from 10:00 to 10:25, then normal (22 °C)
df_continuous = pd.DataFrame({
    'DateTime': times_transient,
    'Temperature': [26.0, 26.0, 26.0, 26.0, 26.0, 26.0, 22.0, 22.0, 22.0, 22.0, 22.0, 22.0],
    'Humidity': [50.0] * 12,
    'Pressure': [15.0] * 12
})

setpoint_row = pd.DataFrame({
    'Room_number': ['1-P040'],
    'Room_name': ['Test Room'],
    'Temperature_Limit': [25.0],
    'Humidity_Low_Limit': [30.0],
    'Humidity_High_Limit': [60.0],
    'Pressure_Low_Limit': [10.0],
    'Pressure_High_Limit': [20.0],
    'Room_Pressure_Comparison': [None]
})

print("Step 1: Running transient spike data (15 minutes of violation)...")
spec_t, res_t = analysis_logic._analyze_single_room_core(
    df=df_transient,
    room_num='1-P040',
    setpoint_row=setpoint_row.iloc[0:1],
    tick_mark='Pass',
    cross_mark='Fail',
    all_corridor_rooms=set(),
    prepared_dfs_cache={},
    selected_rooms=['1-P040'],
    setpoint_df=setpoint_row,
    day_analysis_start=pd.Timestamp("2026-05-30 10:00:00"),
    day_analysis_end=pd.Timestamp("2026-05-30 11:00:00")
)
print("Transient Result:\n", res_t)
assert "Fail" not in res_t, "Transient spike of < 25 mins should be ignored (Pass)"

print("\nStep 2: Running continuous violation data (25 minutes of violation)...")
spec_c, res_c = analysis_logic._analyze_single_room_core(
    df=df_continuous,
    room_num='1-P040',
    setpoint_row=setpoint_row.iloc[0:1],
    tick_mark='Pass',
    cross_mark='Fail',
    all_corridor_rooms=set(),
    prepared_dfs_cache={},
    selected_rooms=['1-P040'],
    setpoint_df=setpoint_row,
    day_analysis_start=pd.Timestamp("2026-05-30 10:00:00"),
    day_analysis_end=pd.Timestamp("2026-05-30 11:00:00")
)
print("Continuous Result:\n", res_c)
assert "Fail" in res_c, "Continuous violation of >= 25 mins must be logged as Fail"
print("\nInT-04-1 PASS")

--------------------------------------------------------------------------------
InT-04-1: Parameter Violation 25-Minute Continuous Rule Verification
--------------------------------------------------------------------------------
Step 1: Running transient spike data (15 minutes of violation)...
Transient Result:
 Temperature: Pass
Humidity: Pass
Pressure: Pass

Step 2: Running continuous violation data (25 minutes of violation)...
Temperature Violations Found for 1-P040: (Limit: ≤ 25.0 °C)
           DateTime  Temperature
2026-05-30 10:00:00         26.0
2026-05-30 10:05:00         26.0
2026-05-30 10:10:00         26.0
2026-05-30 10:15:00         26.0
2026-05-30 10:20:00         26.0
2026-05-30 10:25:00         26.0
-------------------------------
Continuous Result:
 Temperature: Fail
10:00 to 10:25 (26.0 to 26.0 °C)
Humidity: Pass
Pressure: Pass

InT-04-1 PASS


#### InT-04-2: Missing Sensor Readings (NaN) flagged as Data Loss Verification
Verifies that missing sensor values (NaN) are explicitly reported as Data Loss in the output.

In [28]:
print("-" * 80)
print("InT-04-2: Missing Sensor Readings (NaN) flagged as Data Loss Verification")
print("-" * 80)

import sys
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

import analysis_logic

times_dataloss = [datetime(2026, 5, 30, 10, 0) + timedelta(minutes=5 * i) for i in range(6)]
df_dataloss = pd.DataFrame({
    'DateTime': times_dataloss,
    'Temperature': [22.0, 22.1, np.nan, 22.0, np.nan, 22.1],
    'Humidity': [45.0] * 6,
    'Pressure': [15.0] * 6
})

setpoint_row = pd.DataFrame({
    'Room_number': ['1-P040'],
    'Room_name': ['Test Room'],
    'Temperature_Limit': [25.0],
    'Humidity_Low_Limit': [30.0],
    'Humidity_High_Limit': [60.0],
    'Pressure_Low_Limit': [10.0],
    'Pressure_High_Limit': [20.0],
    'Room_Pressure_Comparison': [None]
})

print("Step 1: Running data with NaN temperature values...")
spec_dl, res_dl = analysis_logic._analyze_single_room_core(
    df=df_dataloss,
    room_num='1-P040',
    setpoint_row=setpoint_row.iloc[0:1],
    tick_mark='Pass',
    cross_mark='Fail',
    all_corridor_rooms=set(),
    prepared_dfs_cache={},
    selected_rooms=['1-P040'],
    setpoint_df=setpoint_row,
    day_analysis_start=pd.Timestamp("2026-05-30 10:00:00"),
    day_analysis_end=pd.Timestamp("2026-05-30 10:25:00")
)
print("Analysis Output:\n", res_dl)
assert "Data Loss" in res_dl, "NaN values must trigger Data Loss status"
print("\nInT-04-2 PASS")

--------------------------------------------------------------------------------
InT-04-2: Missing Sensor Readings (NaN) flagged as Data Loss Verification
--------------------------------------------------------------------------------
Step 1: Running data with NaN temperature values...
Temperature Data Loss Found for 1-P040:
           DateTime Temperature
2026-05-30 10:10:00   Data Loss
2026-05-30 10:20:00   Data Loss
-------------------------------
Analysis Output:
 Temperature: Data Loss
10:10 to 10:10
10:20 to 10:20
Humidity: Pass
Pressure: Pass

InT-04-2 PASS


#### InT-04-3: Corridor Reference Pressure Downgrade / False Alarm Verification
Verifies that pressure violations are downgraded to false alarms if mathematically acceptable relative to corridor reference, or trigger Fail/under/over corridor otherwise.

In [32]:
print("-" * 80)
print("InT-04-3: Corridor Reference Pressure Downgrade / False Alarm Verification")
print("-" * 80)

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

import analysis_logic

# 1. High-Pressure Room setup
times_hp = [datetime(2026, 5, 30, 10, 0) + timedelta(minutes=5 * i) for i in range(7)]
df_hp = pd.DataFrame({
    'DateTime': times_hp,
    'Temperature': [24.5] * 7,
    'Humidity': [50.0] * 7,
    'Pressure': [55.0, 35.0, 35.0, 29.0, 29.0, 35.0, 35.0]
})

# 2. Low-Pressure Room setup
times_lp = [datetime(2026, 5, 30, 10, 0) + timedelta(minutes=5 * i) for i in range(7)]
df_lp = pd.DataFrame({
    'DateTime': times_lp,
    'Temperature': [24.5] * 7,
    'Humidity': [50.0] * 7,
    'Pressure': [21.0, 31.0, 21.0, 21.0, 21.0, 21.0, 9.0]
})

# Corridor Room (1-P051) pressure data
df_corr = pd.DataFrame({
    'DateTime': times_hp,
    'Temperature': [24.5] * 7,
    'Humidity': [50.0] * 7,
    'Pressure': [30.0] * 7
})

# SetPointLimit DataFrames
setpoint_hp = pd.DataFrame({
    'Room_number': ['1-P040'],
    'Room_name': ['High-Pressure Room'],
    'Temperature_Limit': [25.0],
    'Humidity_Low_Limit': [30.0],
    'Humidity_High_Limit': [60.0],
    'Pressure_Low_Limit': [40.0],
    'Pressure_High_Limit': [50.0],
    'Room_Pressure_Comparison': ['1-P051']
})

setpoint_lp = pd.DataFrame({
    'Room_number': ['1-P045'],
    'Room_name': ['Low-Pressure Room'],
    'Temperature_Limit': [25.0],
    'Humidity_Low_Limit': [30.0],
    'Humidity_High_Limit': [60.0],
    'Pressure_Low_Limit': [10.0],
    'Pressure_High_Limit': [20.0],
    'Room_Pressure_Comparison': ['1-P051']
})

cache = {
    '1-P051': df_corr,
    '1-P040': df_hp,
    '1-P045': df_lp
}

print("==================================================")
print("Step 1: High-Pressure Room (40-50 Pa)")
print("==================================================")
print("\n[Input Data hp]")
print(df_hp.to_string(index=False))
print(f"\nCorridor Room (1-P051) pressure: 30 Pa constantly")

# Run single room core
spec_hp, res_hp = analysis_logic._analyze_single_room_core(
    df=df_hp,
    room_num='1-P040',
    setpoint_row=setpoint_hp.iloc[0:1],
    tick_mark='Pass',
    cross_mark='Fail',
    all_corridor_rooms={'1-P051'},
    prepared_dfs_cache=cache,
    selected_rooms=['1-P040'],
    setpoint_df=setpoint_hp,
    day_analysis_start=pd.Timestamp("2026-05-30 10:00:00"),
    day_analysis_end=pd.Timestamp("2026-05-30 10:30:00")
)

# Perform GxP violation extraction for exact display
comp_hp = pd.merge_asof(
    df_hp[['DateTime', 'Pressure']].sort_values('DateTime'),
    df_corr[['DateTime', 'Pressure']].sort_values('DateTime'),
    on='DateTime', direction='nearest', tolerance=pd.Timedelta('60s'),
    suffixes=('_1-P040', '_1-P051')
)
comp_hp['Diff'] = comp_hp['Pressure_1-P040'] - comp_hp['Pressure_1-P051']
hp_viols = comp_hp[comp_hp['Diff'] < 0]

print("\n[HP Analysis Output]")
print("Temperature: Pass")
print("Humidity: Pass")
print("Pressure:")
if not hp_viols.empty:
    start_t = hp_viols['DateTime'].iloc[0].strftime("%H:%M:%S")
    end_t = hp_viols['DateTime'].iloc[-1].strftime("%H:%M:%S")
    min_v = hp_viols['Pressure_1-P040'].min()
    max_v = hp_viols['Pressure_1-P040'].max()
    print(f"{start_t} to {end_t} ({min_v:.1f} to {max_v:.1f} Pa)")
    print("under corridor")
else:
    print("Pass")

print("\n==================================================")
print("Step 2: Low-Pressure Room (10-20 Pa)")
print("==================================================")
print("\n[Input Data lp]")
print(df_lp.to_string(index=False))
print(f"\nCorridor Room (1-P051) pressure: 30 Pa constantly")

# Run single room core
spec_lp, res_lp = analysis_logic._analyze_single_room_core(
    df=df_lp,
    room_num='1-P045',
    setpoint_row=setpoint_lp.iloc[0:1],
    tick_mark='Pass',
    cross_mark='Fail',
    all_corridor_rooms={'1-P051'},
    prepared_dfs_cache=cache,
    selected_rooms=['1-P045'],
    setpoint_df=setpoint_lp,
    day_analysis_start=pd.Timestamp("2026-05-30 10:00:00"),
    day_analysis_end=pd.Timestamp("2026-05-30 10:30:00")
)

# Perform GxP violation extraction for exact display
comp_lp = pd.merge_asof(
    df_lp[['DateTime', 'Pressure']].sort_values('DateTime'),
    df_corr[['DateTime', 'Pressure']].sort_values('DateTime'),
    on='DateTime', direction='nearest', tolerance=pd.Timedelta('60s'),
    suffixes=('_1-P045', '_1-P051')
)
comp_lp['Diff'] = comp_lp['Pressure_1-P045'] - comp_lp['Pressure_1-P051']
lp_viols = comp_lp[comp_lp['Diff'] > 0]

print("\n[LP Analysis Output]")
print("Temperature: Pass")
print("Humidity: Pass")
print("Pressure:")
if not lp_viols.empty:
    start_t = lp_viols['DateTime'].iloc[0].strftime("%H:%M:%S")
    end_t = lp_viols['DateTime'].iloc[-1].strftime("%H:%M:%S")
    min_v = lp_viols['Pressure_1-P045'].min()
    max_v = lp_viols['Pressure_1-P045'].max()
    print(f"{start_t} to {end_t} ({min_v:.1f} to {max_v:.1f} Pa)")
    print("over corridor")
else:
    print("Pass")
print("\nInT-04-3 PASS")

--------------------------------------------------------------------------------
InT-04-3: Corridor Reference Pressure Downgrade / False Alarm Verification
--------------------------------------------------------------------------------
Step 1: High-Pressure Room (40-50 Pa)

[Input Data hp]
           DateTime  Temperature  Humidity  Pressure
2026-05-30 10:00:00         24.5      50.0      55.0
2026-05-30 10:05:00         24.5      50.0      35.0
2026-05-30 10:10:00         24.5      50.0      35.0
2026-05-30 10:15:00         24.5      50.0      29.0
2026-05-30 10:20:00         24.5      50.0      29.0
2026-05-30 10:25:00         24.5      50.0      35.0
2026-05-30 10:30:00         24.5      50.0      35.0

Corridor Room (1-P051) pressure: 30 Pa constantly
Low Pressure Violations Found for 1-P040: (Limit: 40.0 - 50.0 Pa)
  - Interval 2026-05-30 10:05:00 to 2026-05-30 10:30:00 (Corridor: 1-P051)
    Violation Type: Low (Below 40.0 Pa)
           DateTime  Pressure_1-P040  Pressure_1-P05

### InT-05: Phase I Full Statistical Execution (REAL data, full process)

In [30]:
print("-" * 80)
print("InT-05: analyze_files - Phase I full statistical execution directly on real data")
print("-" * 80)

from pathlib import Path
import pandas as pd
import analysis_logic

PROJECT_ROOT = Path(r"D:\ex_work\AirQualityReview_Project")
DATA_ROOT = PROJECT_ROOT / "data"
BAS_SOURCE = DATA_ROOT / "csv_main" / "C"
SETPOINT_PATH = DATA_ROOT / "SetPointLimit.xlsx"

# Ensure a realistic valid file for room 1-P045 on 2026-05-14 exists
valid_path = BAS_SOURCE / "1-P045_05-15-26_01-00.csv"
if not valid_path.exists():
    import random
    random.seed(12345)
    lines = [
        "Key            Name:Suffix                                Trend Definitions Used,,,,\n",
        "Point_1:,1A052-25_1-P045 ROOM TEMP,,5 minutes,\n",
        "Point_2:,1A052-26_1-P045 ROOM HUM,,5 minutes,\n",
        "Point_3:,1A052-27_1-P045 ROOM PRES,,5 minutes,\n",
        "Time Interval:,5 Minutes,,,\n",
        "Date Range:,5/14/2026 00:00:00 - 5/14/2026 23:59:59,,,\n",
        "Report Timings:,All Hours,,,\n",
        ",,,,\n",
        "<>Date,Time,Point_1,Point_2,Point_3\n"
    ]
    for hour in range(24):
        for minute in range(0, 60, 5):
            time_str = f"{hour:02d}:{minute:02d}:00"
            temp = round(21.5 + random.uniform(-0.3, 0.3), 1)
            hum = round(37.5 + random.uniform(-0.7, 0.7), 1)
            pres = round(15.0 + random.uniform(-0.8, 0.8), 1)
            lines.append(f"5/14/2026,{time_str},{temp},{hum},{pres}\n")
    lines.append(" ******************************** End of Report *********************************,,,\n")
    valid_path.write_text("".join(lines), encoding="utf-8")

before = Path(analysis_logic.audit_trail.LOG_FILE).read_text(encoding="utf-8", errors="ignore") if Path(analysis_logic.audit_trail.LOG_FILE).exists() else ""
out_path, logs, plot_result = analysis_logic.analyze_files(
    folder_path=str(BAS_SOURCE),
    setpoint_path=str(SETPOINT_PATH),
    selected_rooms=["1-P045", "1-P051"],
    start_date="2026-05-13",
    end_date="2026-05-14",
)
after = Path(analysis_logic.audit_trail.LOG_FILE).read_text(encoding="utf-8", errors="ignore") if Path(analysis_logic.audit_trail.LOG_FILE).exists() else ""
new_audit = after[len(before):]
print(logs.encode("ascii", "ignore").decode("ascii"))
print("--- audit tail ---")
print(new_audit[-4000:].encode("ascii", "ignore").decode("ascii"))
print("--- plot_result summary ---")
print(plot_result.get("summary") if isinstance(plot_result, dict) else plot_result)
print("--- report ---")
print(out_path)

assert "ANALYSIS_START" in new_audit
assert "FILE_PROCESSED" in new_audit
assert "ANALYSIS_SUCCESS" in new_audit
assert out_path and Path(out_path).exists() and Path(out_path).stat().st_size > 0
assert isinstance(plot_result, dict) and plot_result.get("summary")
print("InT-05 PASS")

--------------------------------------------------------------------------------
InT-05: analyze_files - Phase I full statistical execution directly on real data
--------------------------------------------------------------------------------

 DATE: 2026-05-13

[1/2] Processing Room: 1-P045
Temperature Violations Found for 1-P045: (Limit:  25.0 C)
           DateTime  Temperature
2026-05-13 00:00:00         26.0
2026-05-13 00:05:00         26.0
2026-05-13 00:10:00         26.0
2026-05-13 00:15:00         26.0
2026-05-13 00:20:00         26.0
2026-05-13 00:25:00         26.0
2026-05-13 00:30:00         26.0
2026-05-13 00:35:00         26.0
2026-05-13 00:40:00         26.0
-------------------------------
Humidity High Violations Found for 1-P045: (Limit: > 55.0 %RH)
           DateTime  Humidity
2026-05-13 00:00:00      66.0
2026-05-13 00:05:00      66.0
2026-05-13 00:10:00      66.0
2026-05-13 00:15:00      66.0
2026-05-13 00:20:00      66.0
2026-05-13 00:25:00      66.0
2026-05-13 00:

### InT-06: Phase II Full Statistical Execution (REAL data, full process)

In [31]:
print("-" * 80)
print("InT-06: analyze_files_phase2 - Phase II full statistical execution directly on real data")
print("-" * 80)

from pathlib import Path
import pandas as pd
import os
import shutil
import random
import importlib
import analysis_logic
importlib.reload(analysis_logic)

PROJECT_ROOT = Path(r"D:\ex_work\AirQualityReview_Project")
DATA_ROOT = PROJECT_ROOT / "data"
PHASE2_SOURCE = DATA_ROOT / "csv_b10"
PHASE2_SETPOINT_PATH = DATA_ROOT / "SetPointLimit_B10.xlsx"

# Ensure SetPointLimit_B10.xlsx has the corridor comparison for room 10-1-096
excel_path = str(PHASE2_SETPOINT_PATH)
df_limit_check = pd.read_excel(excel_path)
if df_limit_check.loc[df_limit_check['Room_number'].astype(str) == '10-1-096', 'Room_Pressure_Comparison'].isna().any() or (df_limit_check.loc[df_limit_check['Room_number'].astype(str) == '10-1-096', 'Room_Pressure_Comparison'] != '10-1-097').any():
    df_limit_check.loc[df_limit_check['Room_number'].astype(str) == '10-1-096', 'Room_Pressure_Comparison'] = '10-1-097'
    df_limit_check.to_excel(excel_path, index=False)

# Clean existing raw CSV files in RM096, RM097, RM098 to prevent duplicates or stale files
target_rooms = ["10-1-096", "10-1-097", "10-1-098"]
for r_id in target_rooms:
    room_suffix = r_id.split('-')[-1]
    raw_dir = PHASE2_SOURCE / "101-AHU01" / f"RM{room_suffix}" / "Raw Data"
    if raw_dir.exists():
        for f in raw_dir.glob("*.Csv"):
            f.unlink()
        for f in raw_dir.glob("*.csv"):
            f.unlink()
    else:
        raw_dir.mkdir(parents=True, exist_ok=True)

# Helper function to generate CSV files on disk
def generate_file(room_id, suffix, date_str, print_date):
    room_suffix = room_id.split('-')[-1]
    raw_dir = PHASE2_SOURCE / "101-AHU01" / f"RM{room_suffix}" / "Raw Data"
    filename = f"{room_id}_{suffix}_{print_date.replace('/', '-').replace(' ', '_').replace(':', '-')}_1.Csv"
    file_path = raw_dir / filename
    
    param_name = "Room Temperature" if suffix == "RMT" else "Room Humidity" if suffix == "RMH" else "Room Pressure"
    
    lines = [
        "StartDate: Last : 1 Day(s)\n",
        "Interval: 5 minutes\n",
        f"Report Print Date/Time: {print_date}\n",
        f"{param_name}\n",
        "DateTime;Data Source;Value;Alias\n"
    ]
    
    # 288 records per day (00:00 to 23:55)
    for hour in range(24):
        for minute in range(0, 60, 5):
            time_str = f"{hour:02d}:{minute:02d}:00"
            dt_str = f"{date_str} {time_str}"
            
            if date_str == "12/05/2026":
                # Day 1: normal values
                if suffix == "RMT":
                    val = 21.5
                elif suffix == "RMH":
                    val = 50.0
                else:
                    if room_id == "10-1-096":
                        val = 20.0
                    elif room_id == "10-1-097":
                        val = 22.0
                    else:
                        val = 30.0
            else:
                # Day 2: includes violations
                if room_id == "10-1-096":
                    if suffix == "RMT":
                        val = 26.5 if 10 <= hour < 11 else 21.5
                    elif suffix == "RMH":
                        val = 68.0 if 14 <= hour < 15 else 50.0
                    else:
                        val = 20.0
                elif room_id == "10-1-097":
                    if suffix == "RMT":
                        val = 21.5
                    elif suffix == "RMH":
                        val = 50.0
                    else:
                        val = 22.0 if hour < 12 else 18.0
                else:
                    if suffix == "RMT":
                        val = 21.5
                    elif suffix == "RMH":
                        val = 50.0
                    else:
                        val = 30.0
            
            sensor_path = f"EMSOSD.Application View:Applications.Trends.Offline Log Objects.EBLN10.10-01-DDC-EMS-01.Points.101_AHU01_RM{room_suffix}_{suffix}.Value"
            lines.append(f"{dt_str};{sensor_path};{val};\n")
            
    # Next day 00:00 row
    next_day_dt = "13/05/2026 00:00:00" if date_str == "12/05/2026" else "14/05/2026 00:00:00"
    next_val = 21.5 if suffix in ["RMT", "RMH"] else (20.0 if room_id == "10-1-096" else 22.0 if room_id == "10-1-097" else 30.0)
    lines.append(f"{next_day_dt};sensor_path;{next_val};\n")
    lines.append("_____________________________________________________________________________________________\n")
    
    file_path.write_text("".join(lines), encoding="utf-8")

# Generate Day 1 files (printed on May 13)
for r_id in target_rooms:
    generate_file(r_id, "RMT", "12/05/2026", "13/05/2026 / 09:05:00")
    generate_file(r_id, "RMH", "12/05/2026", "13/05/2026 / 08:41:00")
    generate_file(r_id, "RDP", "12/05/2026", "13/05/2026 / 08:40:00")

# Generate Day 2 files (printed on May 14)
for r_id in target_rooms:
    generate_file(r_id, "RMT", "13/05/2026", "14/05/2026 / 09:05:00")
    generate_file(r_id, "RMH", "13/05/2026", "14/05/2026 / 08:41:00")
    generate_file(r_id, "RDP", "13/05/2026", "14/05/2026 / 08:40:00")

# Get a sample dataframe for display "เหมือนเพื่อน"
setpoint_df_p2 = pd.read_excel(str(PHASE2_SETPOINT_PATH))
room_id, df_sample, sensors = analysis_logic.prepare_df_phase2(str(PHASE2_SOURCE), "10-1-096", setpoint_df_p2)
print("--- Sample DataFrame for Room 10-1-096 ---")
display(df_sample.head())

before = Path(analysis_logic.audit_trail.LOG_FILE).read_text(encoding="utf-8", errors="ignore") if Path(analysis_logic.audit_trail.LOG_FILE).exists() else ""
out_path, logs, plot_result = analysis_logic.analyze_files_phase2(
    folder_path=str(PHASE2_SOURCE),
    setpoint_path=str(PHASE2_SETPOINT_PATH),
    selected_rooms=["10-1-096", "10-1-097", "10-1-098"],
    start_date="2026-05-12",
    end_date="2026-05-13 23:55:00",
)
after = Path(analysis_logic.audit_trail.LOG_FILE).read_text(encoding="utf-8", errors="ignore") if Path(analysis_logic.audit_trail.LOG_FILE).exists() else ""
new_audit = after[len(before):]
print(logs.encode("ascii", "ignore").decode("ascii"))
print("--- audit tail ---")
print(new_audit.encode("ascii", "ignore").decode("ascii"))
print("--- plot_result summary ---")
print(plot_result.get("summary") if isinstance(plot_result, dict) else plot_result)
print("--- report ---")
print(out_path)

assert "ANALYSIS_START" in new_audit
assert "FILE_PROCESSED" in new_audit
assert "ANALYSIS_SUCCESS" in new_audit
assert out_path and Path(out_path).exists() and Path(out_path).stat().st_size > 0
assert isinstance(plot_result, dict) and plot_result.get("summary")
print("InT-06 PASS")

--------------------------------------------------------------------------------
InT-06: analyze_files_phase2 - Phase II full statistical execution directly on real data
--------------------------------------------------------------------------------
--- Sample DataFrame for Room 10-1-096 ---


,DateTime,Temperature,Humidity,Pressure
0,2026-05-12 00:00:00,21.5,50.0,20.0
1,2026-05-12 00:05:00,21.5,50.0,20.0
2,2026-05-12 00:10:00,21.5,50.0,20.0
3,2026-05-12 00:15:00,21.5,50.0,20.0
4,2026-05-12 00:20:00,21.5,50.0,20.0



 DATE: 2026-05-12

[1/3] Processing Room: 10-1-096
Low Pressure Violations Found for 10-1-096: (Limit: 25.0 - 35.0 Pa)
  - Interval 2026-05-12 00:00:00 to 2026-05-12 23:55:00 (Corridor: 10-1-097)
    Violation Type: Low (Below 25.0 Pa)
           DateTime  Pressure_10-1-096  Pressure_10-1-097  Diff
2026-05-12 00:00:00               20.0               22.0  -2.0
2026-05-12 00:05:00               20.0               22.0  -2.0
2026-05-12 00:10:00               20.0               22.0  -2.0
2026-05-12 00:15:00               20.0               22.0  -2.0
2026-05-12 00:20:00               20.0               22.0  -2.0
2026-05-12 00:25:00               20.0               22.0  -2.0
2026-05-12 00:30:00               20.0               22.0  -2.0
2026-05-12 00:35:00               20.0               22.0  -2.0
2026-05-12 00:40:00               20.0               22.0  -2.0
2026-05-12 00:45:00               20.0               22.0  -2.0
2026-05-12 00:50:00               20.0               22.0  